# Section 5.3 - Concentration and Allocation Structure

This notebook builds the paper-ready figures for the concentration and allocation section.

Core questions:
1. How concentrated are the portfolios?
2. Are the allocations meaningfully different?
3. Do simpler methods produce more balanced allocations?
4. Do hierarchical methods introduce structure?
5. Does concentration help explain the instability observed in Section 5.2?

Main data sources:
- `concentration.csv`
- one representative run's `weights.csv` files


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

pio.renderers.default = "plotly_mimetype"

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "portafolios").exists():
    raise RuntimeError("Could not locate the repository root containing the portafolios package.")

OUTPUT_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "data_exports" / "final_experimental_setup",
    PROJECT_ROOT / "outputs" / "final_experimental_setup",
]
OUTPUT_DIR = next((path for path in OUTPUT_CANDIDATES if path.exists()), None)
if OUTPUT_DIR is None:
    raise RuntimeError("Could not locate final experimental setup outputs.")

TABLE_DIR = OUTPUT_DIR / "tables"
RUNS_DIR = OUTPUT_DIR / "framework_runs"
FIGURE_DIR = OUTPUT_DIR / "paper_figures" / "section_5_3"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

concentration = pd.read_csv(TABLE_DIR / "concentration.csv")
backtest = pd.read_csv(TABLE_DIR / "backtest_summary.csv")

METHOD_ORDER = [
    "equal_weight",
    "naive_risk_parity",
    "markowitz",
    "hrp_recursive",
    "hca_deprado_ew_nrp",
]

METHOD_LABELS = {
    "equal_weight": "Equal Weight (1/N)",
    "naive_risk_parity": "Naive Risk Parity",
    "markowitz": "Markowitz",
    "hrp_recursive": "HRP Recursive",
    "hca_deprado_ew_nrp": "HCA De Prado EW/NRP",
}

PALETTE = {
    "equal_weight": "#4C78A8",
    "naive_risk_parity": "#72B7B2",
    "markowitz": "#F58518",
    "hrp_recursive": "#54A24B",
    "hca_deprado_ew_nrp": "#E45756",
}


def method_label(method: str) -> str:
    return METHOD_LABELS.get(method, method)


def apply_paper_layout(fig: go.Figure, *, title: str, width: int = 1050, height: int = 620) -> go.Figure:
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=width,
        height=height,
        font=dict(size=13),
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=70, r=40, t=90, b=65),
    )
    return fig


def save_figure(fig: go.Figure, filename: str) -> Path:
    output_path = FIGURE_DIR / filename
    fig.write_html(output_path, include_plotlyjs=True, full_html=True)
    display(fig)
    return output_path


conc = concentration.copy()
conc["method_label"] = conc["method"].map(method_label)

bt = backtest.copy()
bt["run_id"] = (
    bt["universe_id"].astype(str)
    + "_"
    + pd.to_datetime(bt["construction_date"]).dt.strftime("%Y%m%d")
    + "_"
    + bt["estimation_months"].astype(str)
    + "m"
)

print(f"Using output directory: {OUTPUT_DIR}")
print(f"Concentration rows: {len(conc)}")
print(f"Figures will be written to: {FIGURE_DIR}")

Using output directory: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup
Concentration rows: 135
Figures will be written to: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_3


## Writing Flow For The Section

Use this exact progression in the paper:
1. How concentrated are they?
2. What do they look like?
3. Why are they different?
4. How does this help explain the performance results from Section 5.2?


## Summary Table For Interpretation

This table is useful for answering the main concentration question before moving to the figures.

In [2]:
conc_summary = (
    conc.groupby(["method", "method_label"], as_index=False)
    .agg(
        mean_herfindahl=("herfindahl", "mean"),
        std_herfindahl=("herfindahl", "std"),
        mean_max_weight=("max_weight", "mean"),
        std_max_weight=("max_weight", "std"),
    )
    .sort_values("mean_herfindahl", ascending=False)
)
conc_summary

,method,method_label,mean_herfindahl,std_herfindahl,mean_max_weight,std_max_weight
3,markowitz,Markowitz,0.3289,0.1950,0.4420,0.2095
2,hrp_recursive,HRP Recursive,0.1288,0.0698,0.2461,0.1042
1,hca_deprado_ew_nrp,HCA De Prado EW/NRP,0.0839,0.0417,0.1734,0.0554
4,naive_risk_parity,Naive Risk Parity,0.0506,0.0414,0.0829,0.0718
0,equal_weight,Equal Weight (1/N),0.0401,0.0282,0.0401,0.0282


## Figure 1 - Herfindahl Distribution By Method

This directly answers: how concentrated are the portfolios?

In [3]:
fig_conc = px.box(
    conc,
    x="method_label",
    y="herfindahl",
    color="method_label",
    points="all",
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
)
fig_conc.update_traces(jitter=0.22, pointpos=0, marker=dict(size=6, opacity=0.55))
fig_conc.update_xaxes(title="Method")
fig_conc.update_yaxes(title="Herfindahl index")
apply_paper_layout(fig_conc, title="Portfolio Concentration by Method (Herfindahl Distribution)", height=620)
save_figure(fig_conc, "figure_01_herfindahl_boxplot.html")

WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/data_exports/final_experimental_setup/paper_figures/section_5_3/figure_01_herfindahl_boxplot.html')

## Figure 2 - Representative Allocation Structure

One representative run is enough here. The purpose is not to show every run, but to make the allocation logic visually concrete.

Selection rule used below:
- choose the run with the highest average Sharpe ratio across methods, as in Section 5.2.


In [4]:
run_quality = (
    bt.groupby("run_id", as_index=False)["sharpe_ratio"]
    .mean()
    .sort_values("sharpe_ratio", ascending=False)
)
RUN_ID = run_quality.iloc[0]["run_id"]
print("Selected representative run:", RUN_ID)
display(run_quality.head(10))

Selected representative run: sp100_style_top100_20200601_24m


,run_id,sharpe_ratio
22,sp100_style_top100_20200601_24m,2.5558
23,sp100_style_top100_20200601_36m,2.5320
21,sp100_style_top100_20200601_12m,2.4609
3,djia_20200601_12m,2.1073
4,djia_20200601_24m,2.0708
5,djia_20200601_36m,2.0482
12,multi_asset_etf_20200601_12m,1.6053
14,multi_asset_etf_20200601_36m,1.5970
13,multi_asset_etf_20200601_24m,1.5376
9,multi_asset_etf_20190601_12m,1.1698


In [5]:
weight_frames = []
for method in METHOD_ORDER:
    path = RUNS_DIR / RUN_ID / "constructions" / method / "weights.csv"
    if not path.exists():
        continue
    df = pd.read_csv(path)
    if df.shape[1] >= 2:
        df.columns = ["asset", "weight"]
    else:
        raise ValueError(f"Unexpected weights format in {path}")
    df["method"] = method
    df["method_label"] = method_label(method)
    weight_frames.append(df)

weights_long = pd.concat(weight_frames, ignore_index=True)
weights_long = weights_long.sort_values(["method", "weight"], ascending=[True, False])
weights_long.groupby("method_label").head(5)

,asset,weight,method,method_label
0,AAPL,0.0101,equal_weight,Equal Weight (1/N)
1,ABBV,0.0101,equal_weight,Equal Weight (1/N)
2,ABT,0.0101,equal_weight,Equal Weight (1/N)
3,ACN,0.0101,equal_weight,Equal Weight (1/N)
4,ADBE,0.0101,equal_weight,Equal Weight (1/N)
432,GILD,0.1140,hca_deprado_ew_nrp,HCA De Prado EW/NRP
455,MO,0.0620,hca_deprado_ew_nrp,HCA De Prado EW/NRP
471,PM,0.0620,hca_deprado_ew_nrp,HCA De Prado EW/NRP
397,ABBV,0.0447,hca_deprado_ew_nrp,HCA De Prado EW/NRP
414,BMY,0.0447,hca_deprado_ew_nrp,HCA De Prado EW/NRP


The figure below compares the same selected assets across all methods within one representative run. This keeps the weights directly comparable instead of using a different top-asset list for each method.

In [6]:
asset_order = (
    weights_long.groupby("asset", as_index=False)["weight"]
    .max()
    .sort_values("weight", ascending=False)
    .head(12)["asset"]
    .tolist()
)

weights_top = weights_long[weights_long["asset"].isin(asset_order)].copy()
weights_top["asset_label"] = pd.Categorical(
    weights_top["asset"], categories=asset_order, ordered=True
)
weights_top = weights_top.sort_values(["asset_label", "method"])

fig_weights = px.bar(
    weights_top,
    x="asset_label",
    y="weight",
    color="method_label",
    barmode="group",
    category_orders={
        "asset_label": asset_order,
        "method_label": [method_label(m) for m in METHOD_ORDER],
    },
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
)
fig_weights.update_xaxes(title="Asset", tickangle=45)
fig_weights.update_yaxes(title="Weight")
apply_paper_layout(fig_weights, title=f"Representative Allocation Comparison - {RUN_ID}", width=1250, height=700)
save_figure(fig_weights, "figure_02_representative_weights_example.html")

WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/data_exports/final_experimental_setup/paper_figures/section_5_3/figure_02_representative_weights_example.html')

## Interpretation Prompts By Question

Q1. How concentrated are the portfolios?
- Use the Herfindahl boxplot and the average max weight values.
- In the current results, `Markowitz` is by far the most concentrated, while `Equal Weight` is the least concentrated.

Q2. Are the allocations meaningfully different?
- Use the representative weights plot.
- If the top weights are visibly different across methods, then the methods are not just mathematically distinct; they are economically distinct.

Q3. Do simpler methods produce more balanced allocations?
- Compare `Equal Weight`, `Naive Risk Parity`, and `Markowitz`.
- In the current results, yes: the simpler methods are clearly more balanced than `Markowitz`.

Q4. Do hierarchical methods introduce structure?
- Compare `HRP Recursive` and `HCA` to `Markowitz` and the simpler rules.
- The expected reading is that hierarchical methods produce smoother allocations, less extreme than `Markowitz`, but still more structured than `1/N`.

Q5. Does concentration explain instability in Section 5.2?
- Link the high concentration of `Markowitz` back to its weaker and more fragile out-of-sample performance.
- This is the bridge from allocation structure to realized performance variability.


## Section Closing Template

A concise concluding statement for the section can be built like this:

- `The concentration results show that the methods do not merely differ in formula, but in economically meaningful allocation structure.`
- `Simpler rules such as Equal Weight and Naive Risk Parity produce the most balanced portfolios, while Markowitz generates much more concentrated allocations.`
- `The hierarchical methods lie between these extremes, introducing visible structure without reaching the same concentration levels as Markowitz.`
- `This helps explain the Section 5.2 results: higher concentration is associated with greater instability, especially in the case of Markowitz.`


## Generated Files

In [7]:
generated = sorted(FIGURE_DIR.glob("*.html"))
pd.DataFrame({"file": [p.name for p in generated], "path": [str(p) for p in generated]})

,file,path
0,figure_01_herfindahl_boxplot.html,C:\Users\narro\Desktop\semestre\honores_actuar...
1,figure_02_representative_weights_example.html,C:\Users\narro\Desktop\semestre\honores_actuar...
